In [0]:
import requests
import datetime
import json

In [0]:
class Collector:
    def __init__(self, url):
        self.url = url
        self.instance_name = url.strip('/').split('/')[-1] # instance_name

    def get_endpoint(self, **kwargs):
        # url = 'https://pokeapi.co/api/v2/pokemon'

        resp = requests.get(self.url, params=kwargs)

        return resp

    def save_pokemons(self, data):
        now = datetime.datetime.now().strftime("%Y-%m-%d_%H:%M:%S.%f")
        data['ingestion_date'] = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")
        filename = f'/dbfs/mnt/datalake/pokemon/{self.instance_name}/{now}.json'

        with open(filename, 'w') as open_file:
            json.dump(data, open_file)
    
    def get_and_save(self, **kwargs):
        resp = self.get_endpoint(**kwargs)

        if resp.status_code == 200:
            data = resp.json()
            self.save_pokemons(data)
        else:
            return []

    def auto_exec(self, limit=100):
        offset=0
        while True:
            print(f'Offset: {offset}')
            data = self.get_and_save(limit=limit, offset=offset)
            
            if data['next'] == None:
                break
            
            offset += limit

In [0]:
url = 'https://pokeapi.co/api/v2/pokemon'

collector = Collector(url)

resp = collector.auto_exec()

resp

In [0]:
# ajustar a pasta
# dbutils.fs.mkdirs('/mnt/datalake/pokemon/pokemon')

# verificar o conteudo
# dbutils.fs.ls('/mnt/datalake/pokemon/pokemon')

# limpar pasta
dbutils.fs.rm('/mnt/datalake/pokemon/pokemon', True)

# ler as informacoes
df = spark.read.json('/mnt/datalake/pokemon/pokemon')
df.display()

In [0]:
%sql
select
  count,
  ingestion_date,
  -- explode(results) as poke
  poke.*
from pokemons as t1
lateral view explode(results) as poke